# Experiment: ASH_COATED_OSMIUM Local Backtest

Objective:
- Build a conservative local backtest for `ASH_COATED_OSMIUM` that is fast to rerun after parameter changes.
- Compare the same first-draft regime market maker under strict and loose fill assumptions.
- Use the outputs to identify robust parameter regions before iterating on the official IMC website.


In [1]:
# Setup: imports and notebook state
from __future__ import annotations

from dataclasses import asdict, replace
from datetime import datetime
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backtest import (
    ExecutionProfile,
    MMParams,
    build_feature_frame,
    build_report,
    load_round1_product,
    run_backtest,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## Plan

- Strategy direction: keep candidate `D` as the local control, then test one more capacity step, a quote-shape retune, and conservative micro-takers.
- Candidate ladder: evaluate `D`, `E`, `F`, and `G` locally under both execution profiles, then rank by `strict` pnl.
- Metrics to prioritize: `strict_net_pnl`, `worst_day_pnl`, `fill_count`, `passive_share`, `taker_fills`, and `avg_signed_mid_move_5`.
- Manual IMC reruns: use the same `candidate_label` ordering for the website submissions.


In [2]:
# Config: active-round candidate ladder with D as control
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "ash_coated_osmium"
RUN_LABEL = datetime.now().strftime("%Y%m%d_%H%M%S")

PRODUCT = "ASH_COATED_OSMIUM"
BASE_PARAMS = MMParams(
    strategy_name="ASH_COATED_OSMIUM_RegimeMM",
    anchor_lookback=30,
    base_half_spread=4.0,
    inventory_skew=0.40,
    imbalance_skew=1.0,
    dislocation_threshold=5.0,
    defensive_widening_multiplier=2.0,
    max_quote_size=5.0,
    inventory_soft_limit=12.0,
    inventory_hard_limit=20.0,
    thin_depth_threshold=8.0,
    defensive_size_fraction=0.5,
    strong_dislocation_buffer=2.0,
    aggressive_size=0.0,
    enable_dislocation_takers=False,
    dislocation_one_sided_only=True,
)
D_PARAMS = replace(
    BASE_PARAMS,
    strategy_name="ASH_COATED_OSMIUM_Candidate_D",
    base_half_spread=5.0,
    inventory_skew=0.20,
    defensive_widening_multiplier=1.5,
    max_quote_size=10.0,
    inventory_soft_limit=24.0,
    inventory_hard_limit=40.0,
)
CANDIDATE_PARAMS = {
    "D": D_PARAMS,
    "E": replace(
        D_PARAMS,
        strategy_name="ASH_COATED_OSMIUM_Candidate_E",
        max_quote_size=15.0,
        inventory_soft_limit=36.0,
        inventory_hard_limit=60.0,
    ),
    "F": replace(
        D_PARAMS,
        strategy_name="ASH_COATED_OSMIUM_Candidate_F",
        defensive_widening_multiplier=1.25,
        max_quote_size=15.0,
        inventory_soft_limit=36.0,
        inventory_hard_limit=60.0,
    ),
    "G": replace(
        D_PARAMS,
        strategy_name="ASH_COATED_OSMIUM_Candidate_G",
        defensive_widening_multiplier=1.25,
        max_quote_size=15.0,
        inventory_soft_limit=36.0,
        inventory_hard_limit=60.0,
        strong_dislocation_buffer=4.0,
        aggressive_size=2.0,
        enable_dislocation_takers=True,
    ),
}
EXECUTION_PROFILES = {
    "strict": ExecutionProfile(
        name="strict",
        inventory_limit=20.0,
        passive_order_size=5.0,
        taker_order_size=5.0,
        queue_ahead_frac=0.50,
        allow_move_through=False,
        taker_penalty_ticks=1.0,
        signal_delay_steps=1,
        near_limit_frac=0.75,
        force_flat_at_day_end=True,
    ),
    "loose": ExecutionProfile(
        name="loose",
        inventory_limit=20.0,
        passive_order_size=5.0,
        taker_order_size=5.0,
        queue_ahead_frac=0.25,
        allow_move_through=True,
        taker_penalty_ticks=0.5,
        signal_delay_steps=1,
        near_limit_frac=0.75,
        force_flat_at_day_end=True,
    ),
}

CANDIDATE_EXECUTION_PROFILES = {
    "D": {
        "strict": replace(EXECUTION_PROFILES["strict"], inventory_limit=40.0, passive_order_size=10.0, taker_order_size=10.0),
        "loose": replace(EXECUTION_PROFILES["loose"], inventory_limit=40.0, passive_order_size=10.0, taker_order_size=10.0),
    },
    "E": {
        "strict": replace(EXECUTION_PROFILES["strict"], inventory_limit=60.0, passive_order_size=15.0, taker_order_size=15.0),
        "loose": replace(EXECUTION_PROFILES["loose"], inventory_limit=60.0, passive_order_size=15.0, taker_order_size=15.0),
    },
    "F": {
        "strict": replace(EXECUTION_PROFILES["strict"], inventory_limit=60.0, passive_order_size=15.0, taker_order_size=15.0),
        "loose": replace(EXECUTION_PROFILES["loose"], inventory_limit=60.0, passive_order_size=15.0, taker_order_size=15.0),
    },
    "G": {
        "strict": replace(EXECUTION_PROFILES["strict"], inventory_limit=60.0, passive_order_size=15.0, taker_order_size=2.0),
        "loose": replace(EXECUTION_PROFILES["loose"], inventory_limit=60.0, passive_order_size=15.0, taker_order_size=2.0),
    },
}

candidate_config_view = pd.DataFrame(
    [{"candidate_label": label, **asdict(params)} for label, params in CANDIDATE_PARAMS.items()]
)
candidate_profile_view = pd.DataFrame(
    [
        {"candidate_label": label, "profile": profile_name, **asdict(profile)}
        for label, profiles in CANDIDATE_EXECUTION_PROFILES.items()
        for profile_name, profile in profiles.items()
    ]
)

display(candidate_config_view)
display(candidate_profile_view)


,candidate_label,strategy_name,anchor_lookback,base_half_spread,inventory_skew,imbalance_skew,dislocation_threshold,defensive_widening_multiplier,max_quote_size,inventory_soft_limit,inventory_hard_limit,thin_depth_threshold,defensive_size_fraction,strong_dislocation_buffer,aggressive_size,enable_dislocation_takers,dislocation_one_sided_only
0,D,ASH_COATED_OSMIUM_Candidate_D,30,5.0,0.2,1.0,5.0,1.50,10.0,24.0,40.0,8.0,0.5,2.0,0.0,False,True
1,E,ASH_COATED_OSMIUM_Candidate_E,30,5.0,0.2,1.0,5.0,1.50,15.0,36.0,60.0,8.0,0.5,2.0,0.0,False,True
2,F,ASH_COATED_OSMIUM_Candidate_F,30,5.0,0.2,1.0,5.0,1.25,15.0,36.0,60.0,8.0,0.5,2.0,0.0,False,True
3,G,ASH_COATED_OSMIUM_Candidate_G,30,5.0,0.2,1.0,5.0,1.25,15.0,36.0,60.0,8.0,0.5,4.0,2.0,True,True


,candidate_label,profile,name,inventory_limit,passive_order_size,taker_order_size,queue_ahead_frac,allow_move_through,taker_penalty_ticks,signal_delay_steps,near_limit_frac,force_flat_at_day_end
0,D,strict,strict,40.0,10.0,10.0,0.50,False,1.0,1,0.75,True
1,D,loose,loose,40.0,10.0,10.0,0.25,True,0.5,1,0.75,True
2,E,strict,strict,60.0,15.0,15.0,0.50,False,1.0,1,0.75,True
3,E,loose,loose,60.0,15.0,15.0,0.25,True,0.5,1,0.75,True
4,F,strict,strict,60.0,15.0,15.0,0.50,False,1.0,1,0.75,True
5,F,loose,loose,60.0,15.0,15.0,0.25,True,0.5,1,0.75,True
6,G,strict,strict,60.0,15.0,2.0,0.50,False,1.0,1,0.75,True
7,G,loose,loose,60.0,15.0,2.0,0.25,True,0.5,1,0.75,True


## Load Data And Build Features

This step loads the Round 1 files from `data/`, filters to `ASH_COATED_OSMIUM`, and builds the minimal feature frame used by the replay engine and diagnostics.


In [3]:
quotes, trades = load_round1_product(DATA_DIR, PRODUCT)
feature_df = build_feature_frame(quotes, trades)

dataset_overview = pd.DataFrame(
    [
        {
            "product": PRODUCT,
            "quote_rows": len(quotes),
            "trade_rows": len(trades),
            "days": quotes["day"].nunique(),
            "timestamp_min": int(quotes["timestamp"].min()),
            "timestamp_max": int(quotes["timestamp"].max()),
            "both_sides_share": float((feature_df["book_state"] == "both_sides").mean()),
        }
    ]
)
display(dataset_overview)
display(feature_df.head())


,product,quote_rows,trade_rows,days,timestamp_min,timestamp_max,both_sides_share
0,ASH_COATED_OSMIUM,30000,1265,3,0,999900,0.921467


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,best_bid,best_ask,top_bid_depth,top_ask_depth,total_bid_depth_3lvl,total_ask_depth_3lvl,both_sides,bid_only,ask_only,book_empty,book_state,mid_from_touch,clean_mid,spread,top_depth,imbalance_top,imbalance_3lvl,trade_qty_at_bid,trade_qty_at_ask,next_bid_price_1,next_bid_price_2,next_bid_price_3,next_ask_price_1,next_ask_price_2,next_ask_price_3,next_bid_volume_1,next_bid_volume_2,next_bid_volume_3,next_ask_volume_1,next_ask_volume_2,next_ask_volume_3,next_best_bid,next_best_ask,next_clean_mid,fwd_clean_mid_1,fwd_mid_move_1,fwd_clean_mid_5,fwd_mid_move_5,fwd_clean_mid_10,fwd_mid_move_10,day_end
0,-2,0,ASH_COATED_OSMIUM,NaN,NaN,NaN,NaN,NaN,NaN,10010.0,25.0,NaN,NaN,NaN,NaN,10010.0,0.0,NaN,10010.0,0.0,25.0,0.0,25.0,False,False,True,False,ask_only,NaN,NaN,NaN,25.0,-1.0,-1.0,0.0,0.0,9992.0,NaN,NaN,10008.0,10011.0,NaN,15.0,NaN,NaN,15.0,20.0,NaN,9992.0,10008.0,10000.0,10000.0,NaN,10000.0,NaN,9999.5,NaN,False
1,-2,100,ASH_COATED_OSMIUM,9992.0,15.0,NaN,NaN,NaN,NaN,10008.0,15.0,10011.0,20.0,NaN,NaN,10000.0,0.0,9992.0,10008.0,15.0,15.0,15.0,35.0,True,False,False,False,both_sides,10000.0,10000.0,16.0,30.0,0.0,-0.4,0.0,0.0,9992.0,9989.0,NaN,10008.0,10010.0,NaN,15.0,30.0,NaN,15.0,30.0,NaN,9992.0,10008.0,10000.0,10000.0,0.0,9999.5,-0.5,10001.0,1.0,False
2,-2,200,ASH_COATED_OSMIUM,9992.0,15.0,9989.0,30.0,NaN,NaN,10008.0,15.0,10010.0,30.0,NaN,NaN,10000.0,0.0,9992.0,10008.0,15.0,15.0,45.0,45.0,True,False,False,False,both_sides,10000.0,10000.0,16.0,30.0,0.0,0.0,0.0,0.0,9992.0,9989.0,NaN,10008.0,10010.0,NaN,13.0,26.0,NaN,13.0,26.0,NaN,9992.0,10008.0,10000.0,10000.0,0.0,10000.5,0.5,10001.0,1.0,False
3,-2,300,ASH_COATED_OSMIUM,9992.0,13.0,9989.0,26.0,NaN,NaN,10008.0,13.0,10010.0,26.0,NaN,NaN,10000.0,0.0,9992.0,10008.0,13.0,13.0,39.0,39.0,True,False,False,False,both_sides,10000.0,10000.0,16.0,26.0,0.0,0.0,0.0,0.0,9992.0,NaN,NaN,10008.0,10010.0,NaN,15.0,NaN,NaN,15.0,20.0,NaN,9992.0,10008.0,10000.0,10000.0,0.0,10002.0,2.0,10002.5,2.5,False
4,-2,400,ASH_COATED_OSMIUM,9992.0,15.0,NaN,NaN,NaN,NaN,10008.0,15.0,10010.0,20.0,NaN,NaN,10000.0,0.0,9992.0,10008.0,15.0,15.0,15.0,35.0,True,False,False,False,both_sides,10000.0,10000.0,16.0,30.0,0.0,-0.4,0.0,0.0,9992.0,9989.0,NaN,10008.0,10010.0,NaN,13.0,30.0,NaN,13.0,30.0,NaN,9992.0,10008.0,10000.0,10000.0,0.0,9995.5,-4.5,10001.0,1.0,False


## Run Candidate Ladder And Save Reports

This section runs the active-round `D/E/F/G` candidate ladder under both execution profiles, writes reports under `outputs/ash_coated_osmium/<run_label>/<candidate>/<profile>/`, and builds a strict-first ranking table for manual IMC reruns.


In [4]:
RUN_OUTPUT_DIR = OUTPUT_ROOT / RUN_LABEL
candidate_results = {}
candidate_report_paths = {}
candidate_summary_rows = []

for candidate_label, params in CANDIDATE_PARAMS.items():
    candidate_results[candidate_label] = {}
    candidate_profiles = CANDIDATE_EXECUTION_PROFILES.get(candidate_label, EXECUTION_PROFILES)
    for profile_name, profile in candidate_profiles.items():
        result = run_backtest(feature_df, params, profile)
        candidate_results[candidate_label][profile_name] = result
        report_dir = RUN_OUTPUT_DIR / candidate_label / profile_name
        candidate_report_paths[(candidate_label, profile_name)] = build_report(result, report_dir)

        summary_row = result.summary.iloc[0][
            [
                "profile",
                "net_pnl",
                "worst_day_pnl",
                "daily_hit_rate",
                "fill_count",
                "passive_share",
                "taker_fills",
                "buy_fills",
                "sell_fills",
                "max_abs_inventory",
                "near_limit_share",
                "avg_signed_mid_move_5",
                "forced_flat_turnover",
            ]
        ].to_dict()
        candidate_summary_rows.append(
            {
                "candidate_label": candidate_label,
                "anchor_lookback": params.anchor_lookback,
                "base_half_spread": params.base_half_spread,
                "inventory_skew": params.inventory_skew,
                "imbalance_skew": params.imbalance_skew,
                "dislocation_threshold": params.dislocation_threshold,
                "defensive_widening_multiplier": params.defensive_widening_multiplier,
                "enable_dislocation_takers": params.enable_dislocation_takers,
                "dislocation_one_sided_only": params.dislocation_one_sided_only,
                "inventory_limit": profile.inventory_limit,
                "passive_order_size": profile.passive_order_size,
                "taker_order_size": profile.taker_order_size,
                **summary_row,
            }
        )

candidate_summary = pd.DataFrame(candidate_summary_rows)
candidate_summary.to_csv(RUN_OUTPUT_DIR / "candidate_summary.csv", index=False)


## Candidate Ranking

Rank by `strict` pnl first, then inspect the full profile matrix before picking the manual IMC submission order.


In [5]:
candidate_summary_view = (
    candidate_summary[
        [
            "candidate_label",
            "profile",
            "net_pnl",
            "worst_day_pnl",
            "fill_count",
            "passive_share",
            "taker_fills",
            "avg_signed_mid_move_5",
        ]
    ]
    .assign(profile_order=lambda df: df["profile"].map({"strict": 0, "loose": 1}).fillna(99))
    .sort_values(["profile_order", "net_pnl"], ascending=[True, False])
    .drop(columns="profile_order")
    .reset_index(drop=True)
)

strict_ranking = (
    candidate_summary_view.loc[candidate_summary_view["profile"] == "strict"]
    .sort_values("net_pnl", ascending=False)
    .reset_index(drop=True)
)
strict_ranking.insert(0, "strict_rank", strict_ranking.index + 1)
strict_ranking.to_csv(RUN_OUTPUT_DIR / "strict_candidate_ranking.csv", index=False)

report_dir_view = pd.DataFrame(
    [
        {
            "candidate_label": row.candidate_label,
            "strict_rank": int(row.strict_rank),
            "strict_report_dir": str(RUN_OUTPUT_DIR / row.candidate_label / "strict"),
            "loose_report_dir": str(RUN_OUTPUT_DIR / row.candidate_label / "loose"),
        }
        for row in strict_ranking.itertuples(index=False)
    ]
)

display(strict_ranking)
display(candidate_summary_view)
display(report_dir_view)


,strict_rank,candidate_label,profile,net_pnl,worst_day_pnl,fill_count,passive_share,taker_fills,avg_signed_mid_move_5
0,1,F,strict,3664.0,1020.5,279,0.989247,3,-0.648477
1,2,G,strict,3658.0,1036.5,281,0.982206,5,-0.639076
2,3,D,strict,3579.0,958.0,277,0.989170,3,-0.614761
3,4,E,strict,3579.0,958.0,277,0.989170,3,-0.614761


,candidate_label,profile,net_pnl,worst_day_pnl,fill_count,passive_share,taker_fills,avg_signed_mid_move_5
0,F,strict,3664.000,1020.500,279,0.989247,3,-0.648477
1,G,strict,3658.000,1036.500,281,0.982206,5,-0.639076
2,D,strict,3579.000,958.000,277,0.989170,3,-0.614761
3,E,strict,3579.000,958.000,277,0.989170,3,-0.614761
4,F,loose,10397.000,3328.750,1362,0.997797,3,-2.952506
5,G,loose,10332.500,3282.375,1358,0.996318,5,-2.950442
6,D,loose,10255.375,3208.500,1356,0.997788,3,-2.964749
7,E,loose,10246.750,3218.375,1349,0.997776,3,-2.954384


,candidate_label,strict_rank,strict_report_dir,loose_report_dir
0,F,1,/Users/shanilshah/Desktop/coding/imc prosperit...,/Users/shanilshah/Desktop/coding/imc prosperit...
1,G,2,/Users/shanilshah/Desktop/coding/imc prosperit...,/Users/shanilshah/Desktop/coding/imc prosperit...
2,D,3,/Users/shanilshah/Desktop/coding/imc prosperit...,/Users/shanilshah/Desktop/coding/imc prosperit...
3,E,4,/Users/shanilshah/Desktop/coding/imc prosperit...,/Users/shanilshah/Desktop/coding/imc prosperit...


## What To Inspect Before Manual IMC Reruns

Inspect these in order:
- `strict_candidate_ranking.csv`: candidate `D/E/F/G` order and the strict-first scoreboard.
- `<candidate>/strict/summary.csv`: strict net pnl, worst day, passive share, taker fills, and inventory pressure.
- `<candidate>/strict/fills.csv`: for `D/E/F`, verify zero `dislocation_taker_*` reasons; for `G`, verify taker reasons stay limited and small.
- `<candidate>/strict/plots/adverse_selection.png`: confirm whether the quote-shape retune improved post-fill drift materially without broadening taker usage too much.
- `<candidate>/loose/summary.csv`: upside check only after the strict view looks believable.
- Confirm the local profile overrides are in effect: `D` uses `inventory_limit=40`, `passive_order_size=10`, `taker_order_size=10`; `E/F` use `60/15/15`; `G` uses `60/15/2`.
- After official reruns, repoint `ash_backtest_diagnostics.ipynb` to the new IMC JSON outputs and compare `total_pnl`, `max_drawdown`, `dislocation_share`, and `dislocation_mean_pnl_change`.


In [6]:
best_candidate_label = strict_ranking.loc[0, "candidate_label"]
strict_result = candidate_results[best_candidate_label]["strict"]

strict_focus = strict_result.summary[
    [
        "net_pnl",
        "worst_day_pnl",
        "daily_hit_rate",
        "fill_count",
        "passive_share",
        "taker_fills",
        "buy_fills",
        "sell_fills",
        "max_abs_inventory",
        "near_limit_share",
        "avg_signed_mid_move_1",
        "avg_signed_mid_move_5",
        "avg_signed_mid_move_10",
        "forced_flat_turnover",
    ]
].T.rename(columns={0: f"{best_candidate_label}_strict"})

strict_daily_view = strict_result.daily_pnl[
    [
        "day",
        "net_pnl",
        "fill_count",
        "passive_fills",
        "taker_fills",
        "max_abs_inventory",
        "near_limit_share",
        "avg_signed_mid_move_5",
    ]
]

strict_adverse = (
    strict_result.fills.groupby(["liquidity", "side"], observed=True)[
        ["signed_mid_move_1", "signed_mid_move_5", "signed_mid_move_10"]
    ]
    .mean()
    .reset_index()
)

dislocation_taker_reason_count = int(
    strict_result.fills["reason"].fillna("").str.startswith("dislocation_taker").sum()
)
strict_checks = pd.DataFrame(
    [
        {
            "candidate_label": best_candidate_label,
            "strict_net_pnl_ge_3300": bool(strict_result.summary.loc[0, "net_pnl"] >= 3300.0),
            "worst_day_positive": bool(strict_result.summary.loc[0, "worst_day_pnl"] > 0.0),
            "no_dislocation_taker_reasons": dislocation_taker_reason_count == 0,
            "passive_share_ge_0.98": bool(strict_result.summary.loc[0, "passive_share"] >= 0.98),
            "dislocation_taker_reason_count": dislocation_taker_reason_count,
        }
    ]
)

display(pd.DataFrame([{"best_candidate_label": best_candidate_label, "run_output_dir": str(RUN_OUTPUT_DIR)}]))
display(strict_focus)
display(strict_daily_view)
display(strict_adverse)
display(strict_checks)
RUN_OUTPUT_DIR


,best_candidate_label,run_output_dir
0,F,/Users/shanilshah/Desktop/coding/imc prosperit...


,F_strict
net_pnl,3664.000000
worst_day_pnl,1020.500000
daily_hit_rate,1.000000
fill_count,279.000000
passive_share,0.989247
taker_fills,3.000000
buy_fills,136.000000
sell_fills,143.000000
max_abs_inventory,22.000000
near_limit_share,0.000000


,day,net_pnl,fill_count,passive_fills,taker_fills,max_abs_inventory,near_limit_share,avg_signed_mid_move_5
0,-2,1359.5,89.0,88.0,1.0,22.0,0.0,-0.328418
1,-1,1284.0,93.0,92.0,1.0,16.0,0.0,-1.090909
2,0,1020.5,97.0,96.0,1.0,18.5,0.0,-0.513317


,liquidity,side,signed_mid_move_1,signed_mid_move_5,signed_mid_move_10
0,passive,buy,-0.82963,-0.303704,-0.544444
1,passive,sell,-0.91844,-0.968085,-0.975177
2,taker,buy,NaN,NaN,NaN
3,taker,sell,NaN,NaN,NaN


,candidate_label,strict_net_pnl_ge_3300,worst_day_positive,no_dislocation_taker_reasons,passive_share_ge_0.98,dislocation_taker_reason_count
0,F,True,True,True,True,0


PosixPath('/Users/shanilshah/Desktop/coding/imc prosperity/output/jupyter-notebook/ash coated osmium trading/outputs/ash_coated_osmium/20260416_113157')

## Limitations

- The official IMC website backtest is still the source of truth.
- Local passive fills are heuristic because the data does not expose actual queue position or official matching rules.
- The strict profile should drive tuning decisions because it is intentionally harder to pass.
- Day-end force-flat is included to make parameter sets comparable locally; it is not a claim about official evaluation behavior.
